# METAR GeoParquet Analysis and Visualization
---

## Overview
   
Within this notebook, we will cover:

1. Browser-based interactive maps of point-based data 
1. [Geopandas](https://geopandas.org/en/stable/)

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Cartopy Intro](https://foundations.projectpythia.org/core/cartopy/cartopy.html) | Required | Projections and Features |
| [Pandas](https://foundations.projectpythia.org/core/pandas.html) | Required | Tabular Datasets |

- **Time to learn**: 20 minutes
---

## Imports

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from datetime import datetime,timedelta
from dateutil.relativedelta import relativedelta

import geopandas as gpd
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import duckdb

In [2]:
con = duckdb.connect()

In [3]:
duckdb.install_extension("spatial", connection=con)
duckdb.load_extension("spatial", connection=con)

### Explore a monthly METAR Parquet file

In [4]:
sample = gpd.read_parquet(f'yearly/2022-2024_metar.parquet')

In [5]:
sample

,STN,YYMMDD/HHMM,LAT,LON,Elevation,PMSL,ALTI,TMPC,DWPC,SKNT,...,CTYH,P06I,T6XC,T6NC,CEIL,P01I,SNEW,HEAT,WCHT,geometry
Time,,,,,,,,,,,,,,,,,,,,,
2022-01-01 00:00:00+00:00,04V,220101/0000,38.10,-106.17,2385.0,NaN,29.45,0.0,-9.0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.00,25.56,POINT (-106.17 38.1)
2022-01-01 00:00:00+00:00,0CO,220101/0000,39.79,-105.76,3807.0,NaN,29.36,-14.0,-16.0,12.0,...,NaN,NaN,NaN,NaN,5.0,NaN,NaN,6.80,-10.02,POINT (-105.76 39.79)
2022-01-01 00:00:00+00:00,11R,220101/0000,30.22,-96.37,94.0,NaN,29.71,24.5,20.0,9.0,...,NaN,NaN,28.2,24.5,NaN,NaN,NaN,76.98,76.10,POINT (-96.37 30.22)
2022-01-01 00:00:00+00:00,12N,220101/0000,41.01,-74.74,178.0,1014.0,29.93,9.4,8.3,4.0,...,NaN,NaN,10.0,8.3,NaN,NaN,NaN,47.88,47.20,POINT (-74.74 41.01)
2022-01-01 00:00:00+00:00,1A5,220101/0000,35.22,-83.42,616.0,NaN,30.02,12.1,11.6,0.0,...,NaN,0.07,13.6,11.7,10.0,0.05,NaN,53.41,53.78,POINT (-83.42 35.22)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 23:00:00+00:00,1LN,241231/2300,48.79,-102.27,601.0,1023.8,30.10,-12.0,-14.6,20.0,...,NaN,NaN,NaN,NaN,15.0,NaN,NaN,10.40,-9.50,POINT (-102.27 48.79)
2024-12-31 23:00:00+00:00,1MN,241231/2300,48.67,-101.88,569.0,1023.6,30.10,-12.6,-15.1,19.0,...,NaN,NaN,NaN,NaN,16.0,NaN,NaN,9.32,-10.51,POINT (-101.88 48.67)
2024-12-31 23:00:00+00:00,1YT,241231/2300,46.66,-120.45,438.0,1022.3,30.14,2.2,-0.6,0.0,...,NaN,NaN,NaN,NaN,15.0,NaN,NaN,35.96,35.96,POINT (-120.45 46.66)


In [10]:
subset = sample.query ('Time >= "2024-06-25T00:00:00 UTC" & Time <= "2024-06-25T23:00:00 UTC" & TDXC >= 37.8' )

In [11]:
subset['Time'] = subset.index

/knight/jan25/envs/jan26_env/lib/python3.14/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [12]:
sub1 = subset.set_index('STN')

In [13]:
sub1

,YYMMDD/HHMM,LAT,LON,Elevation,PMSL,ALTI,TMPC,DWPC,SKNT,DRCT,...,P06I,T6XC,T6NC,CEIL,P01I,SNEW,HEAT,WCHT,geometry,Time
STN,,,,,,,,,,,,,,,,,,,,,
JWG,240625/0000,35.87,-98.42,472.0,NaN,29.87,35.5,16.7,12.0,170.0,...,NaN,39.0,35.4,NaN,NaN,NaN,96.95,95.90,POINT (-98.42 35.87),2024-06-25 00:00:00+00:00
BPG,240625/0000,32.22,-101.52,784.0,NaN,29.88,37.1,12.7,14.0,170.0,...,NaN,38.0,33.1,NaN,NaN,NaN,97.10,98.78,POINT (-101.52 32.22),2024-06-25 00:00:00+00:00
K82,240625/0000,39.76,-98.79,549.0,NaN,29.71,39.9,14.8,15.0,190.0,...,NaN,43.5,39.5,NaN,NaN,NaN,104.29,103.82,POINT (-98.79 39.76),2024-06-25 00:00:00+00:00
VHN,240625/0000,31.06,-104.78,1206.0,NaN,29.94,38.1,8.6,7.0,80.0,...,NaN,39.0,33.4,NaN,NaN,NaN,97.03,100.58,POINT (-104.78 31.06),2024-06-25 00:00:00+00:00
OJA,240625/0000,35.54,-98.67,489.0,NaN,29.86,35.7,17.6,14.0,190.0,...,NaN,38.6,35.6,NaN,NaN,NaN,98.28,96.26,POINT (-98.67 35.54),2024-06-25 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CPU,240625/0800,38.15,-120.65,404.0,NaN,29.90,23.0,7.7,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,72.20,73.40,POINT (-120.65 38.15),2024-06-25 08:00:00+00:00
QRH,240625/2100,11.55,43.17,10.0,999.2,NaN,37.4,18.6,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,103.12,99.32,POINT (43.17 11.55),2024-06-25 21:00:00+00:00
OKAS,240625/2100,29.35,47.35,144.0,995.9,NaN,36.6,7.6,11.0,300.0,...,NaN,NaN,NaN,NaN,NaN,NaN,94.01,97.88,POINT (47.35 29.35),2024-06-25 21:00:00+00:00


In [15]:
sub1.explore?

Signature: sub1.explore(*args, **kwargs) -> 'folium.Map'
Docstring:
Explore data in interactive map based on GeoPandas and folium/leaflet.js.

Generate an interactive leaflet map based on :class:`~geopandas.GeoDataFrame`

Parameters
----------
column : str, np.array, pd.Series (default None)
    The name of the dataframe column, :class:`numpy.array`,
    or :class:`pandas.Series` to be plotted. If :class:`numpy.array` or
    :class:`pandas.Series` are used then it must have same length as dataframe.
cmap : str, matplotlib.Colormap, branca.colormap or function (default None)
    The name of a colormap recognized by ``matplotlib``, a list-like of colors,
    :class:`matplotlib.colors.Colormap`, a :class:`branca.colormap.ColorMap` or
    function that returns a named color or hex based on the column
    value, e.g.::

        def my_colormap(value):  # scalar value defined in 'column'
            if value > 1:
                return "green"
            return "red"

color : str, array-lik

In [16]:
m = sub1.explore(color='purple')

In [18]:
# Add title
import folium

title_html = """
<h3 align="center" style="font-size:20px">
    <b>Temperatures >= 100 ° F, 6/24/2024</b>
</h3>
"""

m.get_root().html.add_child(folium.Element(title_html))

# Save or display
m.save("map.html")

In [20]:
m

In [ ]:
db = duckdb.read_parquet(f'monthly/concatenated_file.parquet')

In [ ]:
db.sql('SHOW ALL TABLES')

In [ ]:
%%time
sql = """
CREATE TABLE metar AS
SELECT 
    Time, STN, TMPC, DWPC, PMSL, ALTI,
    ST_Point(LON, LAT) as geometry
FROM "s3://us-west-2.opendata.source.coop/ktyle/metar2024/2024_metar.parquet" 
WHERE
    Time BETWEEN '2024-06-29 18:00:00' AND '2024-06-29 18:00:00'
LIMIT 1_000_000;
"""
con.execute(sql)

In [ ]:
%%time
sql = """
CREATE TABLE warmhumid AS
SELECT 
    Time, STN, TMPC, DWPC, PMSL, ALTI,
    ST_Point(LON, LAT) as geometry
FROM "s3://us-west-2.opendata.source.coop/ktyle/metar2024/2024_metar.parquet" 
WHERE
    TMPC >= 27 AND
    TMPC <= 30 AND
    DWPC >= 21 AND
    DWPC <= 28
    
--LIMIT 1_000_000;
"""
db = con.execute(sql)

In [ ]:
url = "s3://us-west-2.opendata.source.coop/ktyle/metar2024/2024_metar.parquet"

In [ ]:
url

In [ ]:
sql = f"""CREATE TABLE nys_dec2024 AS 

In [ ]:
sql0 = """
PREPARE query AS 
  SELECT STN, TMPC, DWPC, PMSL,  "YYMMDD/HHMM" as DATTIM, 
  ST_Point(LON, LAT) as geometry
---FROM "s3://us-west-2.opendata.source.coop/ktyle/metar2024/2024_metar.parquet"
  FROM $url
  WHERE
    LON >= -80 AND
    LAT >= 40 AND
    LON <= -72 AND
    LAT <= 45 AND
    Time BETWEEN '2024-12-01 00:00:00' AND '2024-12-31 23:59:59';
"""

In [ ]:
sql = f"""CREATE TABLE nys_dec2024 AS 
SELECT STN, TMPC, DWPC, PMSL,  "YYMMDD/HHMM" as DATTIM, 
ST_Point(LON, LAT) as geometry
---FROM "s3://us-west-2.opendata.source.coop/ktyle/metar2024/2024_metar.parquet"
FROM $url
WHERE
    LON >= -80 AND
    LAT >= 40 AND
    LON <= -72 AND
    LAT <= 45 AND
    Time BETWEEN '2024-12-01 00:00:00' AND '2024-12-31 23:59:59';
"""

In [ ]:
con.query?

In [ ]:
con.execute(f"{sql0}, {url}")

In [ ]:
db = con.execute(sql,url=$url)

In [ ]:
sql = """
CREATE TABLE warmhumid AS
SELECT 
    STN, TMPC, DWPC, PMSL,  "YYMMDD/HHMM" as DATTIM,
    ST_Point(LON, LAT) as geometry
FROM "yearly/2024_metar.parquet" 
WHERE
    LON >= -74 AND
    LAT >= 42.6 AND
    LON <= -73.5 AND
    LAT <= 42.8 AND
    Time BETWEEN '2024-01-01 00:00:00' AND '2024-12-31 23:59:00'
    LIMIT 1_000_000;
"""
db = con.execute(sql)

In [ ]:
db.sql('SHOW ALL TABLES')

In [ ]:
con.execute("DROP TABLE metar")

In [ ]:
#df = db.df()

In [ ]:
table = con.table("metar")

df = table.df()

df

In [ ]:
df[df['TMPC'] > 24]

In [ ]:
sql = """
CREATE TABLE heat AS
SELECT 
    Time, STN, TMPC, DWPC, HEAT,
    "YYMMDD/HHMM" as DATTIM,
    ST_Point(LON, LAT) as geometry
FROM "monthly/2024_metar.parquet" 
WHERE
    HEAT > 100 AND
    LAT > 40 AND
    LAT < 45 AND
    LON > -80 AND
    LON < -71 AND
    Time BETWEEN '2024-06-01 00:00:00' AND '2024-06-26 00:00:00'
LIMIT 1_000_000;
"""
con.execute(sql)

In [ ]:
sql = """
CREATE TABLE tmpc AS
SELECT 
--    TMPC,
    Time, STN, TMPC, DWPC, HEAT, "YYMMDD/HHMM",
    ST_Point(LON, LAT) as geometry
FROM "monthly/concatenated_file.parquet" 
--FROM "yearly/2024_metar.parquet" 
WHERE
    LON >= -80 AND
    LAT >= 37 AND
    LON <= -65 AND
    LAT <= 45 AND
    Time BETWEEN '2024-11-01 00:00:00' AND '2024-11-26 00:00:00' AND
    TMPC > 28
LIMIT 1_000_000;
"""
con.execute(sql)

In [ ]:
con.execute("DROP TABLE metar")

In [ ]:
layer = ScatterplotLayer.from_duckdb(con.table("metar"), con, radius_min_pixels=2.5)

In [ ]:
m = Map(layer)

May run into a JavaScript error if running from JupyterHub, likely due to lack of `anywidget` package in the Jupyterhub environment. 

In [ ]:
m

In [ ]:
db.close()

In [ ]:
duckdb.query('''
   SELECT *
   FROM 'db'
   WHERE TMPC BETWEEN '22.0' AND '24.0' AND Time BETWEEN '2024-06-23 00:00:00' AND '2024-06-23 02:00:00'
''')

In [ ]:
duckdb.query('''
   SELECT *
   FROM '20240623_metar.parquet'
   WHERE TMPC BETWEEN '22.0' AND '24.0' AND Time BETWEEN '2024-06-23 00:00:00' AND '2024-06-23 02:00:00'
''')

In [ ]:
db.

In [ ]:
viz(sample.loc['2024-06-23T23:00'])

In [ ]:
sample.explore('TMPC')

In [ ]:
gdfWorldMetar.explore(column='TMPC')

<div class="alert alert-info"><b>Note: </b> Most of the non-US data is not available until approximately 15 minutes past each hour. If you do not see many worldwide stations, try re-reading the data file after that point in the hour.</div>

<div class="alert alert-warning"><b>Exercise: </b> Create a color-coded map from one variable in the dataset.</div>

In [ ]:
# Write your code here
gdfWorldMetar.explore(column='TMPC')

<div class="alert alert-info">Note that the color scale is unaffected by any location whose temperature value was missing ... GeoPandas "knows" to exlucde <code>NaN</code> from the range of valid values.</div>

Plot hourly precipitation

In [ ]:
gdfWorldMetar.explore(column='P01M')

<div class="alert alert-info">With few exceptions, hourly precip is only provided by US stations. For this variable, a better visualization would exclude all the missing values (in the US, a missing value denotes <b>no precip</b>, while a <b>trace</b> = 0.00)</div>

In [ ]:
gdfWorldMetar = gdfWorldMetar.dropna(subset=['P01M']).reset_index(drop=True)

In [ ]:
gdfWorldMetar.explore(column='P01M')

## References

1. [GeoPandas](https://geopandas.org)
1. [EPSG](https://epsg.io)